```{contents}
```

## Workflows

GAN = **Generator (G)** + **Discriminator (D)**

* **Generator:** Creates fake samples from noise
* **Discriminator:** Judges whether a sample is real or fake
* Both are trained *simultaneously but adversarially*

$$
\text{Goal: } G^* = \arg \min_G \max_D V(D, G)
$$

---

### Workflow 

#### Prepare Real Data

* Load real samples from a dataset (e.g., MNIST images).
* These samples come from the **true data distribution** $p_{data}(x)$.

🧩 Example:

```python
real_images = next(iter(dataloader))
```

* Each real image = positive sample for the discriminator.

---

#### Generate Fake Data

* Sample **random noise** $z \sim p_z(z)$, often Gaussian or Uniform distribution.
* Pass this noise to the **Generator (G)** to produce fake samples.

$$
x_{fake} = G(z)
$$

🧩 Example:

```python
noise = torch.randn(batch_size, 100)
fake_images = generator(noise)
```

🧠 *Intuition:*
The generator learns to map random noise → realistic-looking data.

---

#### Train the Discriminator (D)

* The discriminator receives **both real and fake data**.
* It outputs a probability (close to 1 for real, 0 for fake).
* Compute discriminator loss as:
  $$
  L_D = -[\log D(x_{real}) + \log(1 - D(G(z)))]
  $$
* Update **D’s parameters** to maximize this loss (better at distinguishing).

🧩 Example:

```python
optimizer_D.zero_grad()
real_loss = criterion(D(real_images), torch.ones_like(real_labels))
fake_loss = criterion(D(fake_images.detach()), torch.zeros_like(fake_labels))
D_loss = real_loss + fake_loss
D_loss.backward()
optimizer_D.step()
```

🧠 *Intuition:*
D becomes a **teacher** that critiques G’s fake data.

---

#### Train the Generator (G)

* The generator wants to **fool** the discriminator.
* Its goal: make $D(G(z)) \to 1$
* Generator loss:
  $$
  L_G = -\log(D(G(z)))
  $$
* Update **G’s parameters** to minimize this loss.

🧩 Example:

```python
optimizer_G.zero_grad()
G_loss = criterion(D(fake_images), torch.ones_like(fake_labels))
G_loss.backward()
optimizer_G.step()
```

🧠 *Intuition:*
G uses D’s feedback to learn *what features make data real*.

---

#### Repeat Adversarial Training

1. Train **D** → improve real/fake detection.
2. Train **G** → improve realism of fake data.
3. Alternate between both per batch.

🧩 Typical ratio:

* 1 generator update for every 1–2 discriminator updates.

This **adversarial loop** continues until both networks reach equilibrium:
$$
D(G(z)) \approx 0.5
$$
meaning D can’t reliably distinguish fake from real → GAN has converged.

---

### Evaluation

GANs don’t have accuracy or loss convergence like normal networks.
Instead, we use visual or statistical measures:

| Metric                               | Purpose                                            |
| ------------------------------------ | -------------------------------------------------- |
| **FID (Fréchet Inception Distance)** | Measures realism and diversity of generated data   |
| **IS (Inception Score)**             | Evaluates sample diversity and clarity             |
| **Visual Check**                     | Plot generated samples to inspect quality manually |

---

### Inference (Generation Phase)

Once trained:

* Freeze the discriminator.
* Use only the generator.
* Input random noise → produce new synthetic data.

$$
x_{new} = G(z)
$$

🧩 Example:

```python
z = torch.randn(1, 100)
generated_image = generator(z)
```

🧠 *Intuition:*
The generator now acts as a *creative artist* — it no longer competes, only generates.

---

### Full Training Loop Summary

| Step | Network       | Input                      | Output                | Loss Function        |
| ---- | ------------- | -------------------------- | --------------------- | -------------------- |
| 1    | Discriminator | Real data                  | Classify as real (1)  | (-\log D(x_{real}))  |
| 2    | Discriminator | Fake data (G(z))           | Classify as fake (0)  | (-\log(1 - D(G(z)))) |
| 3    | Generator     | Noise (z)                  | Fake data (realistic) | (-\log(D(G(z))))     |
| 4    | Repeat        | Alternate G and D training | Both improve          | Minimax loss         |

---

### Workflow Visualization

```
   Step 1: Real Samples  ─────┐
                              ▼
                        [Discriminator]
                              ▲
   Step 2: Fake Samples <────┘
      │
      ▼
   [Generator] ← Random Noise (z)
```

* **Discriminator learns**: “What makes data real?”
* **Generator learns**: “How to fool the discriminator?”
* **Result:** Generator implicitly learns the *true data distribution.*

---

**End Goal**

When training completes:

* Generator produces data **indistinguishable** from real examples.
* Discriminator accuracy hovers near 50% (completely fooled).

$$
p_{model}(x) \approx p_{data}(x)
$$

That means: the generator successfully **learned the data distribution**.